In [ ]:
#Import Json LCOH Heating
import json
from pathlib import Path

dir_Fe = Path(r"C:\Users\patru\PycharmProjects\CatalyticMethanePyrolysis\FeMP\Results_TEA")
dir_AC = Path(r"C:\Users\patru\PycharmProjects\CatalyticMethanePyrolysis\ACMP\Results_TEA")
dir_Ni = Path(r"C:\Users\patru\PycharmProjects\CatalyticMethanePyrolysis\NiMP\Results_TEA")

file_heating_Fe = dir_Fe / "LCOH_heatings.json"
file_heating_AC = dir_AC / "LCOH_heatings.json"
file_heating_Ni = dir_Ni / "LCOH_heatings.json"

with open(file_heating_Fe, "r") as f: heating_Fe = json.load(f)
with open(file_heating_AC, "r") as f: heating_AC = json.load(f)
with open(file_heating_Ni, "r") as f: heating_Ni = json.load(f)

In [ ]:
#Import Json CO2 Heating
import json
from pathlib import Path

dir_Fe = Path(r"C:\Users\patru\PycharmProjects\FeMP\TEA\Ni_CMP")
dir_AC = Path(r"C:\Users\patru\PycharmProjects\CMP\TEA\Ni_CMP")
dir_Ni = Path(r"C:\Users\patru\PycharmProjects\NiMP_Elec_Try1\TEA\Ni_CMP")

file_co2_Fe = dir_Fe / "CO2_results.json"
file_co2_AC = dir_AC / "CO2_results.json"
file_co2_Ni = dir_Ni / "CO2_results.json"

with open(file_co2_Fe, "r") as f: co2_Fe = json.load(f)
with open(file_co2_AC, "r") as f: co2_AC = json.load(f)
with open(file_co2_Ni, "r") as f: co2_Ni = json.load(f)

In [ ]:
# Heating Strategies plot: LCOH + Emissions

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import scienceplots

from pathlib import Path
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MultipleLocator


plt.style.use(["science", "ieee"])

tick_font = 13.5
xlabel_font = 13.0
axis_label_font = 14.8
legend_font = 12.2

mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Times"],
    "font.size": tick_font,
    "axes.labelsize": axis_label_font,
    "xtick.labelsize": xlabel_font,
    "ytick.labelsize": tick_font,
    "legend.fontsize": legend_font,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "text.color": "black",
    "axes.labelcolor": "black",
    "axes.edgecolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "axes.grid": False,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

color_Ni = "#E69F00"
color_Fe = "#7B4AB5"
color_AC = "#D62728"


catalysts = {
    "Fe": {
        "lcoh": heating_Fe,
        "co2": co2_Fe,
        "color": color_Fe,
    },
    "AC": {
        "lcoh": heating_AC,
        "co2": co2_AC,
        "color": color_AC,
    },
    "Ni": {
        "lcoh": heating_Ni,
        "co2": co2_Ni,
        "color": color_Ni,
    },
}

preferred_order = [
    "H2",
    "Electric",
    "CH4_with_CCS",
    "CH4_without_CCS",
]

available_strategies = []

for strategy in preferred_order:
    if any(
        strategy in catalyst_data["lcoh"]
        and strategy in catalyst_data["co2"]
        for catalyst_data in catalysts.values()
    ):
        available_strategies.append(strategy)

for catalyst_data in catalysts.values():
    common_keys = set(catalyst_data["lcoh"]) & set(catalyst_data["co2"])

    for strategy in common_keys:
        if strategy not in available_strategies:
            available_strategies.append(strategy)

if len(available_strategies) == 0:
    raise ValueError("No common LCOH/CO2 heating strategies were found.")


strategy_labels = {
    "H2": r"$\mathrm{H_2}$",
    "Electric": "Electric",
    "CH4_with_CCS": r"$\mathrm{CH_4}$ with CCS",
    "CH4_without_CCS": r"$\mathrm{CH_4}$ without CCS",
}

x_labels = [
    strategy_labels.get(strategy, strategy)
    for strategy in available_strategies
]

x = np.arange(len(available_strategies))


all_lcoh = []
all_co2 = []

for catalyst_data in catalysts.values():
    for strategy in available_strategies:

        if (
            strategy in catalyst_data["lcoh"]
            and strategy in catalyst_data["co2"]
        ):
            all_lcoh.append(
                float(catalyst_data["lcoh"][strategy])
            )

            all_co2.append(
                float(catalyst_data["co2"][strategy])
            )


fig, ax1 = plt.subplots(figsize=(7.2, 4.6))

fig.patch.set_facecolor("white")
ax1.set_facecolor("white")

# LCOH Axis (Left)

for catalyst_name, catalyst_data in catalysts.items():

    color = catalyst_data["color"]

    for i, strategy in enumerate(available_strategies):

        if strategy not in catalyst_data["lcoh"]:
            continue

        if strategy not in catalyst_data["co2"]:
            continue

        ax1.scatter(
            x[i],
            float(catalyst_data["lcoh"][strategy]),
            s=52,
            marker="o",
            facecolor=color,
            edgecolor="black",
            linewidth=0.9,
            zorder=4
        )

ax1.set_ylabel(
    r"LCOH / [US\$\,$\cdot$\,kg$^{\mbox{-}1}$]",
    fontsize=axis_label_font,
    labelpad=9
)

ax1.set_xticks(x)

ax1.set_xticklabels(
    x_labels,
    fontsize=xlabel_font
)

ax1.set_xlim(
    -0.55,
    len(available_strategies) - 0.45
)


lcoh_major_step = 5
lcoh_minor_step = 1

lcoh_ymax = max(
    5,
    np.ceil(
        max(all_lcoh)
        * 1.08
        / lcoh_major_step
    )
    * lcoh_major_step
)

ax1.set_ylim(
    0,
    lcoh_ymax
)

ax1.yaxis.set_major_locator(
    MultipleLocator(
        lcoh_major_step
    )
)

ax1.yaxis.set_minor_locator(
    MultipleLocator(
        lcoh_minor_step
    )
)

# CO2 Emissions Axis (Right)

ax2 = ax1.twinx()

for catalyst_name, catalyst_data in catalysts.items():

    color = catalyst_data["color"]

    for i, strategy in enumerate(available_strategies):

        if strategy not in catalyst_data["lcoh"]:
            continue

        if strategy not in catalyst_data["co2"]:
            continue

        ax2.scatter(
            x[i],
            float(catalyst_data["co2"][strategy]),
            s=125,
            marker="*",
            facecolor=color,
            edgecolor="black",
            linewidth=0.9,
            zorder=5
        )


ax2.set_ylabel(
    r"Specific $\mathrm{CO_2}$ emissions / "
    r"[$\mathrm{kg}_{\mathrm{CO_2}}"
    r"\,\cdot\,"
    r"\mathrm{kg}_{\mathrm{H_2}}^{\mbox{-}1}$]",
    fontsize=axis_label_font,
    labelpad=9
)


co2_major_step = 1.0
co2_minor_step = 0.2

co2_ymax = max(
    1.0,
    np.ceil(
        max(all_co2)
        * 1.08
        / co2_major_step
    )
    * co2_major_step
)

ax2.set_ylim(
    0,
    co2_ymax
)

ax2.yaxis.set_major_locator(
    MultipleLocator(
        co2_major_step
    )
)

ax2.yaxis.set_minor_locator(
    MultipleLocator(
        co2_minor_step
    )
)


ax1.tick_params(
    axis="x",
    which="major",
    direction="in",
    top=True,
    bottom=True,
    length=4.9,
    width=1.1,
    labelsize=xlabel_font,
    colors="black",
    pad=7
)

ax1.tick_params(
    axis="y",
    which="major",
    direction="in",
    left=True,
    right=False,
    length=4.9,
    width=1.1,
    labelsize=tick_font,
    colors="black",
    pad=7
)

ax1.tick_params(
    axis="y",
    which="minor",
    direction="in",
    left=True,
    right=False,
    length=2.8,
    width=0.8,
    colors="black"
)


ax2.tick_params(
    axis="y",
    which="major",
    direction="in",
    left=False,
    right=True,
    length=4.9,
    width=1.1,
    labelsize=tick_font,
    colors="black",
    pad=7
)

ax2.tick_params(
    axis="y",
    which="minor",
    direction="in",
    left=False,
    right=True,
    length=2.8,
    width=0.8,
    colors="black"
)


for spine in ax1.spines.values():
    spine.set_color("black")
    spine.set_linewidth(1.1)

for spine in ax2.spines.values():
    spine.set_color("black")
    spine.set_linewidth(1.1)


catalyst_handles = [
    Patch(
        facecolor=catalysts["Fe"]["color"],
        edgecolor="black",
        linewidth=0.8,
        label="Fe"
    ),

    Patch(
        facecolor=catalysts["AC"]["color"],
        edgecolor="black",
        linewidth=0.8,
        label="AC"
    ),

    Patch(
        facecolor=catalysts["Ni"]["color"],
        edgecolor="black",
        linewidth=0.8,
        label="Ni"
    ),
]

metric_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="None",
        markerfacecolor="white",
        markeredgecolor="black",
        markeredgewidth=0.9,
        markersize=6.2,
        label="LCOH"
    ),

    Line2D(
        [0],
        [0],
        marker="*",
        linestyle="None",
        markerfacecolor="white",
        markeredgecolor="black",
        markeredgewidth=0.9,
        markersize=9.5,
        label=r"Specific $\mathrm{CO_2}$"
    ),
]

legend = fig.legend(
    handles=(
        catalyst_handles
        + metric_handles
    ),
    loc="upper center",
    bbox_to_anchor=(0.5, 0.965),
    ncol=5,
    frameon=False,
    fontsize=legend_font,
    columnspacing=1.25,
    handlelength=1.25,
    handletextpad=0.45
)

for text in legend.get_texts():
    text.set_color("black")

fig.subplots_adjust(
    left=0.12,
    right=0.88,
    bottom=0.18,
    top=0.84
)

# ============================================================
# SAVE AS VECTOR PDF
# ============================================================

fig.savefig("heating_strategies.pdf",
    format="pdf"
)


plt.show()